# Demo 25: Monte Carlo: Episode Returns

**挿入位置：p.31**　｜　**実行環境：Google Colab**

**終了したエピソードの報酬列**だけを用いて、訪問した状態の価値を推定します。同じ5×5迷路を使用し、状態ごとに異なる$G_t$を計算します。

**使い方**：ランタイムで「すべてのセルを実行」した後、スライダー／ボタンで条件を変更します。グラフ中の説明は英語、手順と解説は日本語です。CPUで動作します（Demo 28/29 はPyTorchを使用）。

**報酬の添字**：状態 $s_t$ で行動 $a_t$ を実行した結果得る報酬を $r_t$ と呼び、次状態を $s_{t+1}$ とします。経験は $(s_t,a_t,r_t,s_{t+1})$ です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# All maze-based demos share the same 5x5 world and conventions.
# row 0 = top, column 0 = left. The shortest path S->G has 8 steps.
N=5
START=(4,0); GOAL=(0,4)
WALLS=frozenset({(1,1),(2,1),(2,3),(3,3),(3,4)})
ACTIONS=[(-1,0),(1,0),(0,-1),(0,1)]
ACTION_NAMES=['Up','Down','Left','Right']
STATES=[(i,j) for i in range(N) for j in range(N) if (i,j) not in WALLS]
NONTERMINAL=[s for s in STATES if s!=GOAL]

def transition(s,a, step_cost=-0.04, goal_reward=1.0, walls=WALLS):
    if s==GOAL:return GOAL,0.0,True
    di,dj=ACTIONS[a];ns=(s[0]+di,s[1]+dj)
    if not(0<=ns[0]<N and 0<=ns[1]<N) or ns in walls:ns=s
    done=ns==GOAL
    return ns, goal_reward if done else step_cost, done

def render_maze(ax,values=None,policy=None,route=None,title='',walls=WALLS,fmt='.2f'):
    from matplotlib.patches import Rectangle
    ax.set_xlim(0,N);ax.set_ylim(N,0);ax.set_aspect('equal');ax.axis('off')
    valid=[v for s,v in (values or {}).items() if s not in walls and np.isfinite(v)]
    lo=min(valid) if valid else 0;hi=max(valid) if valid else 1
    for i in range(N):
        for j in range(N):
            s=(i,j)
            if s in walls:color='#626B7B'
            elif s==GOAL:color='#FAD9CA'
            elif s==START:color='#CDECFD'
            elif values is not None and s in values and hi>lo:
                color=plt.cm.YlGnBu(0.15+0.58*(values[s]-lo)/(hi-lo))
            else:color='#FFFFFF'
            ax.add_patch(Rectangle((j,i),1,1,facecolor=color,edgecolor='#8392A6',linewidth=1.1))
            if s in walls:continue
            if s==GOAL:label='G'
            elif values is not None and s in values:label=f'{values[s]:{fmt}}'
            elif s==START:label='S'
            else:label=''
            if label:ax.text(j+.5,i+.49,label,ha='center',va='center',color='#102E57',fontsize=9 if values is not None else 13,fontweight='bold')
            if policy is not None and s in policy and s!=GOAL and values is None:
                ax.text(j+.5,i+.72,'↑↓←→'[policy[s]],ha='center',va='center',color='#1763A8',fontsize=13)
    if route and len(route)>1:
        xx=[s[1]+.5 for s in route];yy=[s[0]+.5 for s in route]
        ax.plot(xx,yy,lw=3,color='#F39B2F',marker='o',markersize=4,alpha=.92)
    ax.set_title(title,fontweight='bold',fontsize=12)

In [ ]:
from collections import deque

def distance_to_goal():
    d={GOAL:0};q=deque([GOAL])
    while q:
        s=q.popleft()
        for delta in ACTIONS:
            n=(s[0]+delta[0],s[1]+delta[1])
            if n in STATES and n not in d:d[n]=d[s]+1;q.append(n)
    return d
DIST=distance_to_goal()

def behavior_action(s,rng,explore=.45):
    # ε-soft shortest-distance behavior policy, NOT the learned optimal policy.
    if rng.random()<explore:return int(rng.integers(4))
    qq=[]
    for a in range(4):
        ns,_,_=transition(s,a);qq.append(DIST.get(ns,999))
    best=[a for a in range(4) if qq[a]==min(qq)]
    return int(rng.choice(best))

def sample_episode(rng,explore=.45,cap=120):
    s=START;records=[]
    for t in range(cap):
        a=behavior_action(s,rng,explore);ns,r,done=transition(s,a)
        records.append((s,a,r,ns))
        s=ns
        if done:break
    return records,done

def returns_from_episode(records,gamma=.9):
    out=np.zeros(len(records));g=0.0
    for t in range(len(records)-1,-1,-1):
        g=records[t][2]+gamma*g;out[t]=g
    return out

r,success=sample_episode(np.random.default_rng(7));gs=returns_from_episode(r)
print('Steps =',len(r),'| goal reached =',success)
print('First 8 state-specific returns:',[(r[i][0],round(gs[i],3)) for i in range(min(8,len(r)))])

In [ ]:
def mc_estimate(episodes=300,gamma=.9,explore=.45,seed=7):
    rng=np.random.default_rng(seed);values={s:0.0 for s in STATES};count={s:0 for s in NONTERMINAL}
    snapshots={};success=[];sample=[]
    milestones=sorted(set([1,5,20,100,episodes]))
    for ep in range(1,episodes+1):
        records,done=sample_episode(rng,explore);rets=returns_from_episode(records,gamma)
        seen=set()
        for i,(s,a,r,ns) in enumerate(records):
            if s in seen:continue
            seen.add(s);count[s]+=1
            values[s]+=(rets[i]-values[s])/count[s] # first-visit arithmetic mean
        success.append(int(done))
        if ep==1:sample=(records,rets)
        if ep in milestones:snapshots[ep]=values.copy()
    return snapshots,count,np.array(success),sample

def show_mc(eps,gamma,noise,seed):
    snaps,count,success,sample=mc_estimate(eps,gamma,noise,seed)
    fig,axs=plt.subplots(1,len(snaps),figsize=(3*len(snaps),3.4))
    if len(snaps)==1:axs=[axs]
    for ax,(k,v) in zip(axs,snaps.items()):render_maze(ax,values=v,title=f'After {k} episodes')
    plt.tight_layout();plt.show()
    records,ret=sample
    print('One episode:',' '.join(f'{s}:{g:.2f}' for (s,_,_,_),g in list(zip(records,ret))[:8]))
    print('Success fraction:',round(success.mean(),3),'| Visits to start:',count[START])

In [ ]:
episodes_w=widgets.IntSlider(value=300,min=20,max=1500,step=20,description='Episodes',continuous_update=False)
gamma_w=widgets.FloatSlider(value=.9,min=.5,max=.99,step=.01,description='γ',continuous_update=False)
explore_w=widgets.FloatSlider(value=.45,min=0,max=1,step=.05,description='Randomness',continuous_update=False)
seed_w=widgets.IntSlider(value=7,min=0,max=100,description='Seed')
run_w=widgets.Button(description='Run Monte Carlo',button_style='primary');out=widgets.Output()
def onrun(_):
    with out:
        clear_output(wait=True);show_mc(episodes_w.value,gamma_w.value,explore_w.value,seed_w.value)
run_w.on_click(onrun)
display(widgets.VBox([widgets.HBox([episodes_w,gamma_w]),widgets.HBox([explore_w,seed_w]),run_w,out]));onrun(None)

### Return の定義と更新式（スライドの $r_t$ 方式）
エピソードの終端を $T$ とすると、各時刻からの収益は
$$G_t=\sum_{k=0}^{T-t-1}\gamma^k r_{t+k}.$$
価値関数は First-visit MC の標本平均で更新します。つまり、同じエピソードの最初の訪問だけを採用し、**状態ごとに異なる $G_t$** を使います。

※ 120ステップで時間切れの場合はその時点を有限エピソード終端として扱います（本来のゴール到達とは区別）。

## Try it yourself / 実験課題
- 1回の試行中は価値表が変化しないことと、終了後に一括更新することを確認する。
- 同一エピソードで異なる時刻のG_tを比較する。
- 試行回数と探索率を変え、推定値のばらつきを比較する。